## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.45 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 14.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              ragas==0.3.0 \
              datasets==4.0.0 \
              evaluate==0.4.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q4_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

# Load the model with llama-cpp-python
# uncomment the below snippet of code if the runtime is connected to GPU (T4-GPU recommended)
llm = Llama(
    model_path=model_path,
    n_ctx=4096,        # context window (tokens) the model can attend to
    n_gpu_layers=40,   # number of transformer layers offloaded to the GPU
    n_batch=512,
    verbose=True,      # Set verbose to True to avoid UnsupportedOperation: fileno error
)

# uncomment the below snippet of code if the runtime is connected to CPU only
# llm = Llama(
#     model_path=model_path,
#     n_ctx=4096,
#     n_cores=-2,
#     verbose=False,
# )

mistral-7b-instruct-v0.2.Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 4.37GB            

mistral-7b-instruct-v0.2.Q4_K_M.gguf: downloading bytes:           |  0.00B            

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loa

#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(response(query_1))


llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      70.11 ms /   128 runs   (    0.55 ms per token,  1825.75 tokens per second)
llama_print_timings: prompt eval time =     379.33 ms /    16 tokens (   23.71 ms per token,    42.18 tokens per second)
llama_print_timings:        eval time =    2887.74 ms /   127 runs   (   22.74 ms per token,    43.98 tokens per second)
llama_print_timings:       total time =    3819.49 ms /   143 tokens




Sepsis is a life-threatening condition that can arise from an infection, and prompt recognition and appropriate management are crucial for improving outcomes. In a critical care unit, the following steps should be taken for managing sepsis:

1. Early recognition: Identify patients at risk of developing sepsis based on clinical suspicion, laboratory results, or vital sign abnormalities.
2. Resuscitation: Administer fluids and vasopressors as needed to maintain adequate tissue perfusion and organ function.
3. Antibiotics: Start broad-spectrum antibiotics as soon


**Observations:**
- Without any grounding context, the base Mistral model answers purely from its pretraining knowledge, so the response tends to be a generic, textbook-style summary of sepsis management (early recognition, blood cultures, broad-spectrum antibiotics, fluids, vasopressors, source control) rather than the exact protocol/terminology used by the Merck Manual.
- The model may omit manual-specific details (e.g., precise drug names, dosing, or bundle timing) since it has no access to the source document.
- At `temperature=0` and the default `max_tokens=128`, the answer is deterministic but may be cut off before covering the full protocol.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(query_2))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      72.78 ms /   128 runs   (    0.57 ms per token,  1758.63 tokens per second)
llama_print_timings: prompt eval time =     296.37 ms /    32 tokens (    9.26 ms per token,   107.97 tokens per second)
llama_print_timings:        eval time =    2888.32 ms /   127 runs   (   22.74 ms per token,    43.97 tokens per second)
llama_print_timings:       total time =    3737.68 ms /   159 tokens




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are some common symptoms of appendicitis:

1. Abdominal pain: The pain usually starts as a mild discomfort around the navel area but eventually moves to the lower right side of the abdomen. The pain may worsen with movement or coughing.
2. Loss of appetite: People with appendicitis may lose their appetite due to abdominal pain or nausea.
3. N


**Observations:**
- Appendicitis and appendectomy are common medical knowledge, so the model is likely to correctly state that surgery is generally required and medicine alone is not curative.
- However, the answer may not reflect the manual's specific nuances (e.g., when antibiotics-first management is considered for uncomplicated cases), since there is no grounding context.
- The response may read as a generic overview rather than a manual-sourced clinical answer.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(query_3))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      86.38 ms /   128 runs   (    0.67 ms per token,  1481.89 tokens per second)
llama_print_timings: prompt eval time =     253.10 ms /    34 tokens (    7.44 ms per token,   134.34 tokens per second)
llama_print_timings:        eval time =    2891.14 ms /   127 runs   (   22.76 ms per token,    43.93 tokens per second)
llama_print_timings:       total time =    3962.96 ms /   161 tokens




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects both men and women. It is characterized by round or oval bald patches that develop suddenly on the scalp. In some cases, it can also affect other areas of the body such as the beard, eyebrows, or eyelashes.

The exact cause of alopecia areata is not fully understood, but it is believed to be an autoimmune condition where the immune system attacks the hair follicles, leading to hair loss. Some possible triggers for this condition


**Observations:**
- The query describes symptoms (patchy hair loss/bald spots) without naming the condition, so the model must first infer that this is likely alopecia areata before suggesting treatments - a step that can introduce inaccuracies without grounding.
- Expect a broad, somewhat generic list of causes and treatments rather than the manual's specific treatment hierarchy (e.g., corticosteroids, immunotherapy).
- This question is a good candidate for comparing against the RAG-based answer later, since descriptive (not diagnosis-named) queries are harder for a context-free LLM.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(query_4))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      67.60 ms /   128 runs   (    0.53 ms per token,  1893.41 tokens per second)
llama_print_timings: prompt eval time =     261.74 ms /    28 tokens (    9.35 ms per token,   106.98 tokens per second)
llama_print_timings:        eval time =    2915.58 ms /   127 runs   (   22.96 ms per token,    43.56 tokens per second)
llama_print_timings:       total time =    3721.57 ms /   155 tokens




A person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:

1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek immediate medical attention. The primary goal is to prevent further damage and ensure the patient's safety. Emergency care may include administering oxygen, controlling bleeding, managing airway and breathing, and monitoring vital signs.
2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage


**Observations:**
- This question (traumatic brain injury) has two parts - temporary vs. permanent impairment - and a default `max_tokens=128` may truncate the answer before both are addressed.
- Expect generic advice (rest, monitoring, surgery for severe cases) rather than the manual's severity-based classification and specific management protocols (e.g., ICP management for severe TBI).

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(response(query_5))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      72.70 ms /   128 runs   (    0.57 ms per token,  1760.66 tokens per second)
llama_print_timings: prompt eval time =     199.57 ms /    35 tokens (    5.70 ms per token,   175.37 tokens per second)
llama_print_timings:        eval time =    2926.14 ms /   127 runs   (   23.04 ms per token,    43.40 tokens per second)
llama_print_timings:       total time =    3675.63 ms /   162 tokens




First and foremost, it is essential to ensure the safety of the injured person. If possible, try to keep them calm and still to prevent further injury or discomfort. If the fracture is open or compound (meaning the bone has pierced the skin), do not attempt to move the person without proper medical assistance as this could cause additional harm.

Once the safety of the individual has been secured, follow these steps:

1. Assess the injury: Check the leg for signs of swelling, bruising, or deformity. Try to determine if there is any pain or numbness


**Observations:**
- The model should give sensible general first-aid advice (immobilize, seek medical care) since this is common knowledge, but is unlikely to reflect the manual's specific fracture classification or orthopedic treatment/recovery guidance.
- The question has three parts (precautions, treatment, recovery/care considerations); with only 128 tokens the answer may address just the first one or two before being cut off.

## Question Answering using LLM with Prompt Engineering

In [ ]:
# Helper to build a Mistral-style instruction prompt combining a role/task instruction with the user question
def build_prompt(instruction, question):
    return f"""[INST] {instruction.strip()}

Question: {question} [/INST]"""

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
instruction_1 = (
    "You are an experienced critical care physician. Answer the following medical question "
    "accurately and concisely. Present the answer as a numbered list of protocol steps."
)
prompt_1 = build_prompt(instruction_1, query_1)

# Combination 1: role prompting + output-format instruction, larger token budget, deterministic decoding
print(response(prompt_1, max_tokens=256, temperature=0, top_p=0.95, top_k=50))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     157.09 ms /   256 runs   (    0.61 ms per token,  1629.67 tokens per second)
llama_print_timings: prompt eval time =     203.59 ms /    56 tokens (    3.64 ms per token,   275.06 tokens per second)
llama_print_timings:        eval time =    5922.47 ms /   255 runs   (   23.23 ms per token,    43.06 tokens per second)
llama_print_timings:       total time =    7529.15 ms /   311 tokens


 1. Recognize and suspect sepsis early based on clinical suspicion, laboratory findings, and severity of illness using criteria from the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score.
2. Initiate resuscitation with intravenous fluids to maintain adequate tissue perfusion, aiming for a mean arterial pressure (MAP) >65 mmHg and central venous oxygen saturation (ScvO2) >70%.
3. Administer broad-spectrum antibiotics within 1 hour of recognition, based on suspected source and local antibiogram data.
4. Provide adequate oxygenation and ventilation, aiming for an arterial oxygen saturation (SpO2) >96% and pH >7.35.
5. Monitor and treat hemodynamic instability with vasopressors, if needed, to maintain MAP >65 mmHg.
6. Correct electrolyte imbalances, acid-base disorders, and coagulation abnormalities.
7. Provide source control, such as drainage or debridement,


**Observations:**
- Role-prompting ("critical care physician") plus an explicit "numbered steps" instruction typically produces a more structured, protocol-like answer than the plain query in the previous section.
- Raising `max_tokens` to 256 (from the default 128) reduces the risk of the answer being cut off mid-list.
- `temperature=0` keeps the ordering and content deterministic across reruns.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
instruction_2 = (
    "You are a medical assistant. Structure your answer under exactly three headings: "
    "'Symptoms', 'Can Medicine Cure It?', and 'Surgical Procedure'. Be specific and factual."
)
prompt_2 = build_prompt(instruction_2, query_2)

# Combination 2: structured-output instruction, slightly higher temperature
print(response(prompt_2, max_tokens=300, temperature=0.2, top_p=0.9, top_k=40))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     179.80 ms /   300 runs   (    0.60 ms per token,  1668.51 tokens per second)
llama_print_timings: prompt eval time =     219.06 ms /    82 tokens (    2.67 ms per token,   374.33 tokens per second)
llama_print_timings:        eval time =    7041.60 ms /   299 runs   (   23.55 ms per token,    42.46 tokens per second)
llama_print_timings:       total time =    8751.66 ms /   381 tokens


 **Symptoms:**

Appendicitis is a inflammatory condition of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The following are common symptoms of appendicitis:

1. Sudden onset of pain in the lower right abdomen, often starting as a mild discomfort that eventually becomes sharp and persistent.
2. Loss of appetite and feeling of nausea or vomiting.
3. Fever, usually low-grade at first but rising as the condition worsens.
4. Abdominal swelling and tenderness, especially in the right lower quadrant.
5. Constipation or diarrhea.
6. Inability to pass gas or have a bowel movement.
7. Pain that worsens when walking or moving around.
8. Pain that radiates to other areas of the body, such as the back or thighs.

**Can Medicine Cure It?**

No, medicine alone cannot cure appendicitis. The inflammation of the appendix cannot be reversed with medication, and the appendix cannot function properly once it becomes inflamed. Antibiotics may be used to treat any ac

**Observations:**
- Explicitly requesting three headings forces the model to directly answer the embedded yes/no sub-question ("Can Medicine Cure It?") instead of glossing over it, which was a risk in the unstructured version.
- A small temperature increase (0.2) adds minor lexical variety without compromising factual consistency.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
instruction_3 = (
    "You are a dermatology consultant. Think step by step: first identify the likely medical "
    "condition and its possible causes, then list evidence-based treatments as bullet points."
)
prompt_3 = build_prompt(instruction_3, query_3)

# Combination 3: chain-of-thought instruction, moderate temperature/top_p/top_k
print(response(prompt_3, max_tokens=256, temperature=0.4, top_p=0.85, top_k=30))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     152.14 ms /   256 runs   (    0.59 ms per token,  1682.67 tokens per second)
llama_print_timings: prompt eval time =     218.00 ms /    77 tokens (    2.83 ms per token,   353.22 tokens per second)
llama_print_timings:        eval time =    6037.32 ms /   255 runs   (   23.68 ms per token,    42.24 tokens per second)
llama_print_timings:       total time =    7710.31 ms /   332 tokens


 Possible Causes of Sudden Patchy Hair Loss (Alopecia Areata):

1. Autoimmune disorder: The most common cause of sudden patchy hair loss is an autoimmune disorder called Alopecia Areata, where the immune system attacks hair follicles.
2. Stress: Emotional or physical stress can trigger Alopecia Areata in some individuals.
3. Genetics: Alopecia Areata can run in families.
4. Vitamin deficiencies: Deficiencies in iron, zinc, or vitamin D can contribute to hair loss.
5. Certain medications: Certain drugs like ACE inhibitors, beta-blockers, retinoids, and anticoagulants have been linked to alopecia areata.
6. Infections: Ringworm infections of the scalp can cause patchy hair loss.

Evidence-Based Treatments for Alopecia Areata:

1. Topical corticosteroids: These medications can help reduce inflammation and promote hair regrowth. They can be applied directly to the affected areas in the form of creams, lotions, or


**Observations:**
- The "think step by step" instruction nudges the model to name the underlying condition (e.g., alopecia areata) before listing treatments, improving the logical structure of the answer.
- Moderate `temperature`/`top_p`/`top_k` (0.4/0.85/30) trades a bit of determinism for more natural phrasing, which suits an open-ended, descriptive question like this one.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
instruction_4 = (
    "You are a neurology specialist. Separate your answer into 'Immediate/Emergency Management' "
    "and 'Long-term Rehabilitation'. End with a one-line disclaimer that this is general information, "
    "not a substitute for professional medical advice."
)
prompt_4 = build_prompt(instruction_4, query_4)

# Combination 4: low temperature, larger token budget for a two-part answer
print(response(prompt_4, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     185.30 ms /   289 runs   (    0.64 ms per token,  1559.65 tokens per second)
llama_print_timings: prompt eval time =     221.39 ms /    87 tokens (    2.54 ms per token,   392.97 tokens per second)
llama_print_timings:        eval time =    6863.49 ms /   288 runs   (   23.83 ms per token,    41.96 tokens per second)
llama_print_timings:       total time =    8647.73 ms /   375 tokens


 Immediate/Emergency Management:
1. Ensure airway patency and provide oxygen support if necessary.
2. Monitor vital signs and prevent complications such as seizures, infections, or hemorrhages.
3. Administer medications to manage symptoms like pain, swelling, or increased intracranial pressure.
4. Provide adequate hydration and nutrition.
5. Perform surgical interventions if needed to remove hematomas or decompress the brain.
6. Provide rehabilitative therapies such as physical, occupational, and speech therapy as soon as medically stable.

Long-term Rehabilitation:
1. Implement individualized rehabilitation programs to address cognitive, communicative, sensory, and motor deficits.
2. Provide supportive care to manage symptoms and prevent complications.
3. Encourage participation in activities to promote functional independence.
4. Utilize assistive devices and technologies as needed.
5. Provide psychological support to address emotional and behavioral changes.
6. Collaborate with fami

**Observations:**
- Splitting the answer into emergency vs. long-term sections plus a disclaimer produces a safer, more clinically organized response than the plain-query version.
- `temperature=0.1` keeps facts consistent while `max_tokens=350` gives enough room to cover both parts of this two-section answer without truncation.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
instruction_5 = (
    "You are a first-aid and orthopedic care expert. Provide a step-by-step checklist covering "
    "immediate first aid, medical treatment, and recovery precautions for the described injury."
)
prompt_5 = build_prompt(instruction_5, query_5)

# Combination 5: checklist framing, moderate temperature/top_p/top_k
print(response(prompt_5, max_tokens=300, temperature=0.3, top_p=0.9, top_k=40))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     166.23 ms /   300 runs   (    0.55 ms per token,  1804.73 tokens per second)
llama_print_timings: prompt eval time =     220.39 ms /    82 tokens (    2.69 ms per token,   372.07 tokens per second)
llama_print_timings:        eval time =    7193.26 ms /   299 runs   (   24.06 ms per token,    41.57 tokens per second)
llama_print_timings:       total time =    8877.23 ms /   381 tokens


 1. Immediate First Aid:
   a. Ensure the safety of both the injured person and yourself. If necessary, call for help or have someone call emergency services (dial 911 or local emergency number).
   b. Keep the injured person as comfortable as possible by helping them lie down on a flat surface with the injured leg elevated to reduce swelling.
   c. Apply a sterile pad or clean cloth over the fracture site to prevent infection and control any bleeding. Do not attempt to realign or manipulate the bone.
   d. Immobilize the leg using a splint or sling to prevent further movement and potential damage. Use materials available on hand such as sticks, clothing, or a hiking pole. Be sure to immobilize above and below the fracture site.
   e. Cover the injured leg with a waterproof or breathable material to protect it from the elements.

2. Medical Treatment:
   a. Once medical help arrives, listen to their instructions carefully. Do not move the injured person unless it is absolutely necessar

**Observations:**
- Checklist framing tends to produce an actionable, easy-to-follow list covering first aid through recovery, addressing the multi-part nature of the question better than a free-form response.
- Across all five combinations, explicit output-structure instructions (headings, numbered lists, checklists) had a more visible effect on answer usability than the parameter changes alone, though larger `max_tokens` was necessary whenever the requested structure had multiple sections.

## Data Preparation for RAG

### Loading the Data

In [ ]:
# uncomment and run the following lines for Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pdf_path = "/content/medical_diagnosis_manual.pdf"

pdf_loader = PyMuPDFLoader(pdf_path)
pages = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")
    print(pages[i].page_content, end="\n")
    print("-" * 100)

Page Number : 1
alienrivero@gmail.com
210WJT5VQU
This file is meant for personal use by alienrivero@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
----------------------------------------------------------------------------------------------------
Page Number : 2
alienrivero@gmail.com
210WJT5VQU
This file is meant for personal use by alienrivero@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
----------------------------------------------------------------------------------------------------
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    ...............................................................................................................................................

#### Checking the number of pages

In [ ]:
len(pages)

4114

### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,
    chunk_overlap=100
)

document_chunks = pdf_loader.load_and_split(text_splitter)
len(document_chunks)

4657

### Embedding

In [ ]:
# https://huggingface.co/spaces/mteb/leaderboard
embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")

/tmp/ipykernel_3481/2930130386.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Vector Database

In [ ]:
persist_directory = "medical_manual_db"

if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)

vectorstore = Chroma.from_documents(
    document_chunks,
    embedding_model,
    persist_directory=persist_directory
)

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

### System and User Prompt Template

In [ ]:
qna_system_message = """
You are an assistant to a medical professional. Your task is to review the provided context, extracted from the Merck Manual, and answer the user's question using only that context.
User input will have the context required by you to answer user questions. This context will begin with the token: ###Context.
The context contains references to specific portions of the medical manual relevant to the user query.

User questions will begin with the token: ###Question.

Please answer only using the context provided in the input. Do not mention anything about the context in your final answer. Do not provide any information that is not backed by the given context.

If the answer is not found in the context, respond "I don't know, this is not covered in the provided medical manual." Do not try to make up an answer.

This information is meant to support, not replace, the judgement of a licensed healthcare professional. Do not provide medical advice beyond what is stated in the context.
"""

qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
print(generate_rag_response(query_1))

/tmp/ipykernel_3481/141734326.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      72.86 ms /   128 runs   (    0.57 ms per token,  1756.79 tokens per second)
llama_print_timings: prompt eval time =    4084.22 ms /  3177 tokens (    1.29 ms per token,   777.87 tokens per second)
llama_print_timings:        eval time =    4074.63 ms /   127 runs   (   32.08 ms per token,    31.17 tokens per second)
llama_print_timings:       total time =    8961.68 ms /  3304 tokens


Based on the context, the protocol for managing sepsis in a critical care unit includes the following steps:
1. Aggressive fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
2. Oxygen administration via mask or nasal prongs.
3. Broad-spectrum antibiotics based on culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normal


**Observations:**
- Grounded in the manual's sepsis chapter, the retrieved chunks should let the model cite the actual clinical bundle (recognition, cultures/labs, broad-spectrum antibiotics, fluids, vasopressors, source control) using manual-specific terminology, which is more precise than the no-context answer earlier.
- If the answer still looks generic, the retriever's `k=3` may not be pulling the exact page(s) discussing sepsis protocol - worth inspecting `retriever.get_relevant_documents(query_1)` directly.
- The system prompt's "I don't know" fallback should prevent hallucinated details that aren't in the retrieved context.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
print(generate_rag_response(query_2))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      69.57 ms /   128 runs   (    0.54 ms per token,  1839.82 tokens per second)
llama_print_timings: prompt eval time =    4199.86 ms /  3187 tokens (    1.32 ms per token,   758.83 tokens per second)
llama_print_timings:        eval time =    4326.12 ms /   127 runs   (   34.06 ms per token,    29.36 tokens per second)
llama_print_timings:       total time =    9269.70 ms /  3314 tokens


Based on the context, the common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs include right lower quadant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior spine). Additional signs include pain felt in the right lower quadrant with palpation of the left lower


**Observations:**
- The retrieved context should confirm surgery (appendectomy) as the standard treatment and describe when non-operative/antibiotic management is considered, grounded directly in the manual rather than general knowledge.
- Because appendicitis is likely covered in one focused chapter, this is a good candidate for a high-groundedness answer, useful as a baseline when comparing against the evaluation scores later.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
print(generate_rag_response(query_3))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      81.29 ms /   128 runs   (    0.64 ms per token,  1574.63 tokens per second)
llama_print_timings: prompt eval time =    4010.17 ms /  3063 tokens (    1.31 ms per token,   763.81 tokens per second)
llama_print_timings:        eval time =    4236.46 ms /   127 runs   (   33.36 ms per token,    29.98 tokens per second)
llama_print_timings:       total time =    9231.84 ms /  3190 tokens


Based on the context provided, the condition being described is alopecia areata, which is characterized by sudden patchy hair loss on the scalp or other areas of the body. The causes of alopecia areata are not clearly understood but are believed to be related to autoimmune disorders and genetic susceptibility. The treatment options for alopecia areata include corticosteroids (injected intradermally or used topically), topical anthralin, minoxidil, or immunotherapy (diphencyprone or squaric acid dibutylester).


**Observations:**
- Because the query describes symptoms rather than naming "alopecia areata" directly, retrieval quality depends heavily on semantic similarity between the description and the manual's wording.
- If the top-k chunks miss the relevant dermatology chapter, the RAG answer may default to "I don't know" per the system prompt - a good signal for whether `k` or the embedding model needs tuning (see the Fine-tuning section below).

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
print(generate_rag_response(query_4))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      72.83 ms /   128 runs   (    0.57 ms per token,  1757.54 tokens per second)
llama_print_timings: prompt eval time =    4061.81 ms /  2995 tokens (    1.36 ms per token,   737.36 tokens per second)
llama_print_timings:        eval time =    4415.64 ms /   127 runs   (   34.77 ms per token,    28.76 tokens per second)
llama_print_timings:       total time =    9246.00 ms /  3122 tokens


Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important.


**Observations:**
- Traumatic brain injury management is typically split across severity levels in the manual; watch whether the default `k=3` retrieves both the acute-management and rehabilitation portions, or only one.
- This is a strong motivating example for increasing `k` and `max_tokens`, tested explicitly in the fine-tuning step below.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
print(generate_rag_response(query_5))

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      69.39 ms /   128 runs   (    0.54 ms per token,  1844.59 tokens per second)
llama_print_timings: prompt eval time =    3706.83 ms /  2698 tokens (    1.37 ms per token,   727.85 tokens per second)
llama_print_timings:        eval time =    4257.51 ms /   127 runs   (   33.52 ms per token,    29.83 tokens per second)
llama_print_timings:       total time =    8659.50 ms /  2825 tokens


Based on the context provided, I don't have enough information to directly answer the user's question about fractured leg during hiking trip, including necessary precautions and treatment steps, care, and recovery. The context focuses on snakebites and their treatment. However, I can suggest some general advice based on the information available in the context regarding first aid and supportive care for injuries.

1. Immediate care: The person should be moved beyond the striking distance of any potential danger, such as other hikers or wildlife. They should be kept warm and reassured while being transported to the nearest medical


**Observations:**
- Fracture care often spans multiple manual sections (first aid, orthopedic management, rehabilitation); the response quality here is a good test case for increasing `k` to capture a broader set of relevant chunks.
- Compared to the no-context and prompt-engineered versions, the RAG answer should ground precautions/treatment/recovery guidance in the manual's actual orthopedic content rather than generic first-aid knowledge.

### Fine-tuning

In [ ]:
# 1) Chunking variation: rebuild the corpus with a smaller chunk_size/overlap and compare retrieval
alt_text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,
    chunk_overlap=50
)
alt_document_chunks = pdf_loader.load_and_split(alt_text_splitter)
print("Original chunking (1000/100) ->", len(document_chunks), "chunks")
print("Alternate chunking (500/50)  ->", len(alt_document_chunks), "chunks")

alt_persist_directory = "medical_manual_db_alt"
if not os.path.exists(alt_persist_directory):
    os.makedirs(alt_persist_directory)

alt_vectorstore = Chroma.from_documents(alt_document_chunks, embedding_model, persist_directory=alt_persist_directory)
alt_retriever = alt_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

# 2) Fine-tuning combinations across chunking, retriever (k), and LLM generation parameters
fine_tune_combinations = [
    {"label": "Baseline chunking (1000/100), k=3, greedy decoding",           "query": query_1, "retriever": retriever,     "k": 3, "max_tokens": 200, "temperature": 0,    "top_p": 0.95, "top_k": 50},
    {"label": "Smaller chunking (500/50), same k, greedy decoding",           "query": query_1, "retriever": alt_retriever, "k": 3, "max_tokens": 200, "temperature": 0,    "top_p": 0.95, "top_k": 50},
    {"label": "Baseline chunking, fewer retrieved chunks (k=2)",              "query": query_2, "retriever": retriever,     "k": 2, "max_tokens": 200, "temperature": 0.2,  "top_p": 0.9,  "top_k": 40},
    {"label": "Baseline chunking, more retrieved chunks (k=6)",               "query": query_3, "retriever": retriever,     "k": 6, "max_tokens": 200, "temperature": 0.1,  "top_p": 0.85, "top_k": 30},
    {"label": "Baseline chunking, larger token budget for multi-part answer", "query": query_4, "retriever": retriever,     "k": 4, "max_tokens": 350, "temperature": 0.3,  "top_p": 0.9,  "top_k": 40},
    {"label": "Smaller chunking, broad retrieval, moderate creativity",       "query": query_5, "retriever": alt_retriever, "k": 6, "max_tokens": 300, "temperature": 0.15, "top_p": 0.92, "top_k": 45},
]

original_retriever = retriever

for i, combo in enumerate(fine_tune_combinations, start=1):
    retriever = combo["retriever"]  # swap the global retriever used inside generate_rag_response
    print(f"Combination {i}: {combo['label']}")
    print(f"  k={combo['k']}, max_tokens={combo['max_tokens']}, temperature={combo['temperature']}, top_p={combo['top_p']}, top_k={combo['top_k']}")
    print(generate_rag_response(
        combo["query"],
        k=combo["k"],
        max_tokens=combo["max_tokens"],
        temperature=combo["temperature"],
        top_p=combo["top_p"],
        top_k=combo["top_k"],
    ))
    print("-" * 100)

retriever = original_retriever  # restore the default retriever used by later cells

Original chunking (1000/100) -> 4657 chunks
Alternate chunking (500/50)  -> 8833 chunks
Combination 1: Baseline chunking (1000/100), k=3, greedy decoding
  k=3, max_tokens=200, temperature=0, top_p=0.95, top_k=50


Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     109.30 ms /   200 runs   (    0.55 ms per token,  1829.81 tokens per second)
llama_print_timings: prompt eval time =    3814.89 ms /  2948 tokens (    1.29 ms per token,   772.76 tokens per second)
llama_print_timings:        eval time =    6448.36 ms /   199 runs   (   32.40 ms per token,    30.86 tokens per second)
llama_print_timings:       total time =   11346.10 ms /  3147 tokens
Llama.generate: prefix-match hit


Based on the context, the protocol for managing sepsis in a critical care unit includes the following steps:
1. Aggressive fluid resuscitation with 0.9% normal saline until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg.
2. Oxygen administration via mask or nasal prongs.
3. Broad-spectrum antibiotics based on culture results.
4. Drainage of abscesses and excision of necrotic tissue.
5. Normalization of blood glucose levels.
6. Replacement-dose corticosteroids.
7. Monitoring of systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2
----------------------------------------------------------------------------------------------------
Combination 2: Smaller chunking (500/50), same k, greedy decoding
  k=3, max_tokens=200, temperature=0, top_p=0.95, top_k=50



llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     125.69 ms /   200 runs   (    0.63 ms per token,  1591.18 tokens per second)
llama_print_timings: prompt eval time =    1530.95 ms /  1152 tokens (    1.33 ms per token,   752.47 tokens per second)
llama_print_timings:        eval time =    5887.20 ms /   199 runs   (   29.58 ms per token,    33.80 tokens per second)
llama_print_timings:       total time =    8794.06 ms /  1351 tokens
Llama.generate: prefix-match hit


Based on the context, the protocol for managing sepsis in a critical care unit includes aggressive fluid resuscitation, antibiotics, surgical excision of infected or necrotic tissues and drainage of pus, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Patients should be monitored frequently for systemic pressure, CVP or PAOP, pulse oximetry, ABGs, blood glucose, lactate, electrolyte levels, renal function, and possibly sublingual PCO2. Urine output should be measured, usually with an indwelling catheter. Fluid resuscitation should be given until CVP reaches 8 mm Hg (10 cm H2O) or PAOP reaches 12 to 15 mm Hg. If a patient remains hypotensive after CVP or PAOP has been
----------------------------------------------------------------------------------------------------
Combination 3: Baseline chunking, fewer retrieved chunks (k=2)
  k=2, max_tokens=200, temperature=0.2, top_p=0.9, top_k=40


...

###Answer
The common symptoms for appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs include right lower quadrant direct and rebound tenderness located at McBurney's point. Additional signs include pain felt in the right lower quadrant with palpation of the left lower quadrant (Rovsing sign), pain caused by passive internal rotation of the flexed thigh (obturator sign), or pain felt with passive extension of the right hip joint (psoas sign). Low-grade fever (rectal temperature 37.7 to 38.3° C) is common. Unfortunately, these classic findings appear in less than 50% of patients. Appendicitis cannot be cured via medicine alone; surgery (
----------------------------------------------------------------------------------------------------
Combination 4: Baseline chunking, more retrieved chunks (k=6)
  k=6, max_tokens=200, t


llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      61.88 ms /   115 runs   (    0.54 ms per token,  1858.47 tokens per second)
llama_print_timings: prompt eval time =    5130.37 ms /  3751 tokens (    1.37 ms per token,   731.14 tokens per second)
llama_print_timings:        eval time =    3994.05 ms /   114 runs   (   35.04 ms per token,    28.54 tokens per second)
llama_print_timings:       total time =    9789.42 ms /  3865 tokens
Llama.generate: prefix-match hit


Based on the context provided, the recommended treatments for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function include ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injury to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion
----------------------------------------------------------------------------------------------------
Combination 6: Smaller chunking, broad retrieval, moderate creativity
  k=6, max_tokens=300, temperature=0.15, top_p=0.92, top_k=45



llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     168.57 ms /   300 runs   (    0.56 ms per token,  1779.63 tokens per second)
llama_print_timings: prompt eval time =    3013.90 ms /  2344 tokens (    1.29 ms per token,   777.73 tokens per second)
llama_print_timings:        eval time =    9395.35 ms /   299 runs   (   31.42 ms per token,    31.82 tokens per second)
llama_print_timings:       total time =   14157.88 ms /  2643 tokens


Based on the context provided, here is the answer:
1. The person should be moved beyond the striking distance of any snakes in the area to prevent further injury or snake bites.
2. They should be transported rapidly to the nearest medical facility for further evaluation and treatment.
3. In the meantime, they should be kept warm and reassured.
4. The fractured leg should be immobilized in a functional position just below heart level.
5. All rings, watches, and constrictive clothing should be removed.
6. If there is a tourniquet already in place, it should remain until they reach the hospital.
7. In the emergency department, they will undergo serial assessment and testing, including measurement of extremity circumference, CBC, coagulation profile, measurement of fibrin degradation products, urinalysis, serum electrolytes, BUN, creatinine, blood typing and cross-matching, ECG, chest x-ray, and CK tests as necessary.
8. They may require antibiotics to prevent infection.
9. The wound shoul

**Observations:**
- **Chunking:** smaller chunks (500/50) roughly double the number of chunks versus the baseline (1000/100). Smaller chunks can improve retrieval precision (less irrelevant text per chunk) but risk splitting a single explanation across chunks, which is why `chunk_overlap` matters.
- **Retriever (`k`):** increasing `k` generally helps multi-faceted questions (Q3, Q4, Q5) that span more than one section of the manual, at the cost of a longer, more diluted context the LLM must summarize within `max_tokens`. Lowering `k` for narrowly-scoped questions (Q2) keeps the context focused.
- **LLM parameters:** raising `max_tokens` for multi-part questions (Q4, Q5) reduces truncation; small increases in `temperature`/`top_p`/`top_k` (0.1-0.3) slightly diversify phrasing without introducing noticeable hallucination, since the answer stays constrained by the retrieved context and the "don't know if not in context" instruction.
- Overall, `k` and chunk size have a larger impact on answer completeness than the LLM sampling parameters for this grounded RAG setup.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context.

Instructions:
- First, write down the steps that are needed to evaluate the answer as per the metric.
- Give a step-by-step explanation for your evaluation, referring to the specific parts of the context and the answer.
- The step-by-step evaluation must be done in no more than 2-3 short sentences.
- Finally, respond with a single integer score (1-5) representing the overall rating, prefixed with "Score: ".
"""

In [ ]:
relevance_rater_system_message = """
You are tasked with rating AI-generated answers to questions posed by users about a medical manual.
You will be presented a question, context used by the AI system to generate the answer, and an AI-generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context, while the AI-generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
Relevance measures how well the answer addresses the main aspects of the question, based on the context provided.

Instructions:
- First, write down the steps that are needed to evaluate the answer as per the metric.
- Give a step-by-step explanation for your evaluation, referring to the specific parts of the question and the answer.
- The step-by-step evaluation must be done in no more than 2-3 short sentences.
- Finally, respond with a single integer score (1-5) representing the overall rating, prefixed with "Score: ".
"""

In [ ]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
groundedness_1, relevance_1 = generate_ground_relevance_response(query_1, max_tokens=350)

print(groundedness_1, end="\n\n")
print(relevance_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     194.51 ms /   338 runs   (    0.58 ms per token,  1737.69 tokens per second)
llama_print_timings: prompt eval time =    4127.55 ms /  3192 tokens (    1.29 ms per token,   773.34 tokens per second)
llama_print_timings:        eval time =   10891.30 ms /   337 runs   (   32.32 ms per token,    30.94 tokens per second)
llama_print_timings:       total time =   17164.05 ms /  3529 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      76.09 ms /   148 runs   (    0.51 ms per token,  1945.12 tokens per second)
llama_print_timings: prompt eval time =    4534.26 ms /  3584 tokens (    1.27 ms per token,   790.43 tokens per second)
llama_print_timings:        eval time =    4931.08 ms /   148 runs   (   33.32 ms per token,    30.01 tokens per second)
llama_print_timings:       to

 Steps for evaluation:
1. Identify if all elements of the sepsis management protocol mentioned in the context are included in the answer.
2. Verify that the information in the answer is derived solely from the context provided.

Explanation:
The answer correctly lists all components of sepsis management protocol as per the context, including fluid resuscitation, antibiotics, surgical intervention, supportive care, intensive control of blood glucose, corticosteroids, activated protein C, monitoring of various parameters, and use of vasopressors if necessary. The answer is derived entirely from the information in the context, so the metric is followed completely.

Score: 5

 The AI-generated answer follows the metric mostly (Score: 4). The answer summarizes the key components of managing sepsis in a critical care unit as stated in the Merck Manual context, including fluid resuscitation, antibiotics, surgical intervention, supportive care, monitoring, and potential use of vasopressors and

**Observations:**
- Expect a high groundedness score if the retrieved sepsis-protocol context was on-topic (as seen in the earlier RAG answer); the relevance score should likewise be high since the question is precise and factual, giving the retriever a strong signal.
- If either score is low, check whether the retrieved chunks actually discuss sepsis management, or whether the answer drifted into unsupported claims.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
groundedness_2, relevance_2 = generate_ground_relevance_response(query_2, max_tokens=350)

print(groundedness_2, end="\n\n")
print(relevance_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     201.09 ms /   350 runs   (    0.57 ms per token,  1740.50 tokens per second)
llama_print_timings: prompt eval time =    4364.03 ms /  3425 tokens (    1.27 ms per token,   784.82 tokens per second)
llama_print_timings:        eval time =   11653.02 ms /   349 runs   (   33.39 ms per token,    29.95 tokens per second)
llama_print_timings:       total time =   18246.20 ms /  3774 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      48.15 ms /    95 runs   (    0.51 ms per token,  1973.08 tokens per second)
llama_print_timings: prompt eval time =    5160.42 ms /  3837 tokens (    1.34 ms per token,   743.54 tokens per second)
llama_print_timings:        eval time =    3310.41 ms /    94 runs   (   35.22 ms per token,    28.40 tokens per second)
llama_print_timings:       to

 Steps for evaluation:
1. Identify if the answer mentions symptoms specifically derived from the context.
2. Check if the answer mentions only symptoms for appendicitis without suggesting any treatment or cure.

Evaluation:
The answer mentions common symptoms for appendicitis from the context accurately. It does not suggest any treatment or cure, keeping the metric in mind. Therefore, I would rate this as:
Score: 5.

 Score: 5

The AI-generated answer accurately summarizes the common symptoms of appendicitis from the context and correctly states that surgery is necessary for treatment. It also acknowledges that antibiotics can improve survival rate but are not curative when surgery is impossible. The answer also mentions that resection of the entire mass may be preferable when a large inflammatory mass is found involving the appendix, terminal ileum, and cecum, and that pericolic abscesses can be drained by ultrasound. Overall, the answer is highly relevant to the question.


**Observations:**
- Since appendicitis is well-covered in a single, focused manual chapter, expect both groundedness and relevance to score highly, similar to the pattern seen for narrowly-scoped, well-retrieved questions in the reference RAG case study.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
groundedness_3, relevance_3 = generate_ground_relevance_response(query_3, max_tokens=350)

print(groundedness_3, end="\n\n")
print(relevance_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      77.70 ms /   141 runs   (    0.55 ms per token,  1814.63 tokens per second)
llama_print_timings: prompt eval time =    4390.24 ms /  3301 tokens (    1.33 ms per token,   751.89 tokens per second)
llama_print_timings:        eval time =    4755.12 ms /   140 runs   (   33.97 ms per token,    29.44 tokens per second)
llama_print_timings:       total time =    9948.02 ms /  3441 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     105.19 ms /   208 runs   (    0.51 ms per token,  1977.39 tokens per second)
llama_print_timings: prompt eval time =    4639.81 ms /  3503 tokens (    1.32 ms per token,   754.99 tokens per second)
llama_print_timings:        eval time =    7120.45 ms /   207 runs   (   34.40 ms per token,    29.07 tokens per second)
llama_print_timings:       to

 Steps for evaluation:
1. Identify if the AI-generated answer mentions treatments and causes derived solely from the context.
2. Compare the treatments and causes mentioned in the answer with those in the context.

Evaluation:
The AI-generated answer mentions corticosteroids, topical anthralin, minoxidil, systemic corticosteroids, topical immunotherapy (diphencyprone or squaric acid dibutylester), and psoralen plus ultraviolet A (PUVA) as effective treatments for sudden patchy hair loss. These treatments are all listed in the context as options for treating alopecia areata. The cause of sudden patchy hair loss being an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers is also mentioned in the answer, which is consistent with the information in the context. Therefore, the metric is followed completely.

Score: 5

 Steps for evaluation:
1. Identify the main aspects of the question: effective treatments for sudden patchy hair loss and p

**Observations:**
- This is the best candidate to reveal a groundedness/relevance gap - if the retriever pulled loosely related dermatology content (since the query never names "alopecia areata"), the LLM-judge should assign a lower score with an explanation flagging missing or partially-supported information.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
groundedness_4, relevance_4 = generate_ground_relevance_response(query_4, max_tokens=500)

print(groundedness_4, end="\n\n")
print(relevance_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     103.13 ms /   158 runs   (    0.65 ms per token,  1532.00 tokens per second)
llama_print_timings: prompt eval time =    4281.93 ms /  3233 tokens (    1.32 ms per token,   755.03 tokens per second)
llama_print_timings:        eval time =    5069.29 ms /   157 runs   (   32.29 ms per token,    30.97 tokens per second)
llama_print_timings:       total time =   10529.68 ms /  3390 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      39.27 ms /    70 runs   (    0.56 ms per token,  1782.62 tokens per second)
llama_print_timings: prompt eval time =    4508.44 ms /  3452 tokens (    1.31 ms per token,   765.67 tokens per second)
llama_print_timings:        eval time =    2331.89 ms /    69 runs   (   33.80 ms per token,    29.59 tokens per second)
llama_print_timings:       to

 Score: 5

The AI-generated answer is completely derived from the information presented in the context. It accurately identifies the specific treatments mentioned in the context for a person with traumatic brain injury, including surgery for severe injuries, maintenance of adequate brain perfusion and oxygenation, prevention of complications, and rehabilitation.

 Score: 5

The AI-generated answer directly addresses the main aspects of the question by summarizing the specific treatments mentioned in the context for a person with traumatic brain injury, including surgery for severe injuries, maintenance of adequate brain perfusion and oxygenation, prevention of complications, and rehabilitation.


**Observations:**
- This multi-part question (treatment for temporary vs. permanent impairment) is a good test of relevance - a lower score here would indicate the RAG answer addressed only part of the question, reinforcing the case for the higher `k`/`max_tokens` tested in the fine-tuning step.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
groundedness_5, relevance_5 = generate_ground_relevance_response(query_5, max_tokens=500)

print(groundedness_5, end="\n\n")
print(relevance_5)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =     188.28 ms /   316 runs   (    0.60 ms per token,  1678.35 tokens per second)
llama_print_timings: prompt eval time =    3752.68 ms /  2936 tokens (    1.28 ms per token,   782.38 tokens per second)
llama_print_timings:        eval time =   10177.18 ms /   315 runs   (   32.31 ms per token,    30.95 tokens per second)
llama_print_timings:       total time =   15998.70 ms /  3251 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     379.47 ms
llama_print_timings:      sample time =      41.81 ms /    64 runs   (    0.65 ms per token,  1530.70 tokens per second)
llama_print_timings: prompt eval time =    4322.72 ms /  3313 tokens (    1.30 ms per token,   766.42 tokens per second)
llama_print_timings:        eval time =    2008.40 ms /    63 runs   (   31.88 ms per token,    31.37 tokens per second)
llama_print_timings:       to

 Score: 3

The answer is derived from general medical knowledge related to fractures and supportive care, which is not explicitly stated in the context but can be inferred from the information provided. While not directly derived from the context, the suggestions align with standard precautions and treatment steps for fractures.

 Score: 3
The answer addresses some aspects of the question related to treatment steps for a fractured leg, such as immobilization, pain management, rest, and transportation. However, it does not directly address all the main aspects of the question, which include necessary precautions and care for recovery. The answer also lacks specific information regarding the context of a hiking trip or potential complications from venomous snakebites.


**Observations:**
- Similar to Q4, this question spans first aid, treatment, and recovery; watch whether the relevance judge penalizes the answer for covering only one aspect (e.g., first aid but not recovery precautions) - a useful signal for whether `k` should be increased for broad, multi-topic questions.
- Across all five evaluations, groundedness and relevance scores together give a scalable way to spot which questions need retrieval or prompt tuning, without a clinician manually reviewing every answer.

## Actionable Insights and Business Recommendations

**Key Takeaways**

- A RAG pipeline grounded in the Merck Manual measurably improves answer quality over a bare LLM: retrieved context lets the model cite manual-specific protocols, drug names, and procedures instead of falling back on generic pretraining knowledge, directly reducing the risk of hallucinated or outdated clinical guidance.
- Prompt engineering (role-setting, explicit output structure, step-by-step reasoning instructions) meaningfully improves answer usability even without retrieval, and remains useful on top of RAG to keep answers well-organized (e.g., symptoms vs. treatment, emergency vs. long-term care).
- Retrieval quality is the main lever for answer quality: broader, multi-topic questions (e.g., brain injury, leg fracture) benefit from a higher `k`, while narrow factual questions do better with a smaller, more focused context. `chunk_size`/`chunk_overlap` should be tuned together with `k`, since they jointly determine how much of the source manual is visible to the model for any given answer.
- The "I don't know" instruction embedded in the system prompt is critical for a medical use case: it constrains the assistant to the manual's content and avoids fabricating clinical advice when the manual doesn't cover a topic.
- LLM-as-a-judge evaluation (groundedness + relevance) is a scalable way to continuously monitor answer quality as the manual, chunking strategy, or model is updated, without requiring a clinician to review every response manually.

**Business Recommendations**

- Deploy the RAG assistant as a decision-support tool (not a diagnostic authority) for clinicians and triage staff, paired with a visible disclaimer to consult a licensed professional for final decisions.
- Periodically re-index the vector database whenever the manual is revised, and track groundedness/relevance scores over time as a quality gate before rolling out updates.
- Extend the knowledge base with additional trusted references (e.g., hospital-specific protocols) using the same chunking/embedding/retrieval pipeline, to widen coverage beyond the Merck Manual.
- Invest in retrieval tuning (chunking strategy, embedding model, `k`) ahead of LLM parameter tuning, since the evaluation shows retrieval quality has the larger impact on both groundedness and relevance for this use case.
- Before production use, add a human-in-the-loop review step for high-stakes queries (e.g., treatment decisions), and log low-groundedness/low-relevance responses to drive continuous improvement.

<font size=6 color='blue'>Power Ahead</font>
___